# 👁️ VMamba — Eye Image HB Estimation
**Repo:** [MzeroMiko/VMamba](https://github.com/MzeroMiko/VMamba)  
**Paper:** *VMamba: Visual State Space Models* (ICLR 2025)  
**Key idea:** 2D Selective Scan (SS2D) traverses images in multiple directions (cross-scan).  
**Tasks:** ① Classification ② HB regression


In [ ]:
import subprocess, sys
subprocess.run(["git","clone","--depth=1","https://github.com/MzeroMiko/VMamba.git","VMamba"],check=False)
sys.path.insert(0,"VMamba")
for p in ["mamba-ssm","causal-conv1d","einops","timm","pandas","openpyxl","scikit-learn","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install",p,"-q"],check=False)
print("✅ Done")


In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

CFG = dict(
    image_dir  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye",
    csv_path   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx",
    image_col  = "Patient ID", hb_col="HB", hb_thresh=12.0,
    img_size=224, epochs=30, batch_size=8, lr=1e-4,
    cls_weight=1.0, reg_weight=0.5, val_split=0.2, seed=42,
)


In [ ]:
# Load VMamba backbone and add dual heads
try:
    sys.path.insert(0, "VMamba/classification")
    from models import build_vssm_model
    from config import get_config
    print("✅ VMamba backbone available")
    VMAMBA_OK = True
except Exception as e:
    print(f"Could not load VMamba directly ({e}) — using VMamba-style backbone")
    VMAMBA_OK = False

# ── VMamba dual-head wrapper ──────────────────────────────────────────────────
class VMambaDualHead(nn.Module):
    def __init__(self, num_classes=2, feat_dim=768):
        super().__init__()
        if VMAMBA_OK:
            # Use actual VMamba-T backbone
            import argparse
            cfg = get_config(argparse.Namespace(cfg="VMamba/classification/configs/vssm/vmambav2_tiny_224.yaml",
                             opts=None, batch_size=8, data_path="", zip=False,
                             cache_mode=False, pretrained=None, resume="",
                             accumulation_steps=None, use_checkpoint=False,
                             amp_opt_level=0, output="", tag="", eval=False,
                             throughput=False, fused_layernorm=False, optim=""))
            self.backbone = build_vssm_model(cfg)
            self.backbone.head = nn.Identity()
            feat_dim = self.backbone.num_features
        else:
            from torchvision.models import resnet50
            base = resnet50(weights=None)
            feat_dim = 2048
            base.fc = nn.Identity()
            self.backbone = base
            print("Using ResNet-50 fallback")

        self.cls_head = nn.Linear(feat_dim, num_classes)
        self.reg_head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim//4), nn.GELU(), nn.Linear(feat_dim//4, 1))

    def forward(self, x):
        f = self.backbone(x)
        if f.dim() > 2: f = f.mean([-2,-1])
        return self.cls_head(f), self.reg_head(f)

model = VMambaDualHead().to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")


In [ ]:
# Dataset + Loaders (standard — same as all notebooks)
class EyeHBDataset(Dataset):
    def __init__(self, df, image_dir, image_col, hb_col, hb_thresh, transform=None):
        self.df=df.reset_index(drop=True); self.image_dir=image_dir
        self.image_col=image_col; self.hb_col=hb_col
        self.hb_thresh=hb_thresh; self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; pid=str(row[self.image_col]); hb=float(row[self.hb_col])
        label=0 if hb < self.hb_thresh else 1
        for ext in [".jpg",".jpeg",".png",""]:
            p=os.path.join(self.image_dir, pid+ext)
            if os.path.exists(p): break
        img=Image.open(p).convert("RGB")
        if self.transform: img=self.transform(img)
        return img, torch.tensor(label,dtype=torch.long), torch.tensor([[hb]],dtype=torch.float32)

T_train = transforms.Compose([transforms.Resize((256,256)),transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),transforms.ColorJitter(0.2,0.2,0.1),
    transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
T_val = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

df=pd.read_excel(CFG["csv_path"])
train_df,val_df=train_test_split(df,test_size=CFG["val_split"],random_state=CFG["seed"],
    stratify=(df[CFG["hb_col"]]<CFG["hb_thresh"]).astype(int))
train_loader=DataLoader(EyeHBDataset(train_df,CFG["image_dir"],CFG["image_col"],
    CFG["hb_col"],CFG["hb_thresh"],T_train),batch_size=CFG["batch_size"],shuffle=True,num_workers=2)
val_loader=DataLoader(EyeHBDataset(val_df,CFG["image_dir"],CFG["image_col"],
    CFG["hb_col"],CFG["hb_thresh"],T_val),batch_size=CFG["batch_size"],shuffle=False,num_workers=2)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")


In [ ]:
# Train
optimizer=torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=1e-2)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=CFG["epochs"])
ce_loss=nn.CrossEntropyLoss(); mse_loss=nn.MSELoss()
def loss_fn(lo,hr,lab,hb):
    return CFG["cls_weight"]*ce_loss(lo,lab)+CFG["reg_weight"]*mse_loss(hr,hb)

best=float("inf")
for epoch in range(1,CFG["epochs"]+1):
    model.train(); tl=0
    for imgs,labels,hb in tqdm(train_loader,desc=f"Ep{epoch}",leave=False):
        imgs,labels,hb=imgs.to(DEVICE),labels.to(DEVICE),hb.to(DEVICE)
        optimizer.zero_grad(); lo,hr=model(imgs); loss=loss_fn(lo,hr,labels,hb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step(); tl+=loss.item()
    scheduler.step(); tl/=len(train_loader)
    model.eval(); vl=correct=total=0; hbp=[]; hbt=[]
    with torch.no_grad():
        for imgs,labels,hb in val_loader:
            imgs,labels,hb=imgs.to(DEVICE),labels.to(DEVICE),hb.to(DEVICE)
            lo,hr=model(imgs); vl+=loss_fn(lo,hr,labels,hb).item()
            correct+=(lo.argmax(1)==labels).sum().item(); total+=labels.size(0)
            hbp+=hr.cpu().squeeze().tolist(); hbt+=hb.cpu().squeeze().tolist()
    vl/=len(val_loader)
    if vl<best: best=vl; torch.save(model.state_dict(),"best_vmamba.pth")
    if epoch%5==0 or epoch==1:
        print(f"Ep{epoch:3d}|TL:{tl:.4f}|VL:{vl:.4f}|Acc:{correct/total:.3f}|MAE:{mean_absolute_error(hbt,hbp):.2f}")
